# Revision Questions: Data Understanding & Data Privacy

The data we will work on contains the same kinds of "messiness" that exist in real-world data:
* missing engagement metrics (for "protected" accounts)
* a date column stored as text
* a users table with duplicate rows
* a sentiment file with a messy/mixed dtype
* raw category labels that need to be simplified
* usernames/mentions that should be handled carefully from a privacy perspective

**Files (in the `da3_data` folder):**
* `synthetic_tweets.csv` — one row per tweet
* `synthetic_users_raw.csv` — one row per user *appearance* (authors + mentioned users, so it has duplicates)
* `synthetic_sentiment.csv` — sentiment scores (`positive`, `negative`, `neutral`, SentiStrength-style trinary) for a subset of tweets

## Setup

In [1]:
import pandas as pd
import re

## Question 1 — Load and inspect

Load `synthetic_tweets.csv` into a dataframe called `tweets`. Check its shape, column names, and dtypes.

**Note:** Before working with any data you need to know what you're actually working with. Checking shape/columns/dtypes catches loading mistakes early (wrong file, wrong separator, wrong encoding), problems that are much harder to debug once they've fed into later steps.

In [5]:
# Accessing function documentation without leaving JupyterLab

# 1. help() - the full docstring, printed below this cell. Jupyter.
# help(pd.read_csv)

# # 2. The '?' shortcut - same info as help(), but opens in its own pane (doesn't clutter your notebook)
# pd.read_csv?

# # 3. Shift+Tab - put your cursor inside the parentheses below and press Shift+Tab
# pd.read_csv()

# # this is how you look up documentation for attributes
help(pd.DataFrame.shape)      

Help on property:

    Return a tuple representing the dimensionality of the DataFrame.

    Unlike the `len()` method, which only returns the number of rows, `shape`
    provides both row and column counts, making it a more informative method for
    understanding dataset size.

    See Also
    --------
    numpy.ndarray.shape : Tuple of array dimensions.

    Examples
    --------
    >>> df = pd.DataFrame({"col1": [1, 2], "col2": [3, 4]})
    >>> df.shape
    (2, 2)

    >>> df = pd.DataFrame({"col1": [1, 2], "col2": [3, 4], "col3": [5, 6]})
    >>> df.shape
    (2, 3)



## Question 2 — Check for missing values

Which columns have missing values, and how many? Also check `.describe()` on the numeric columns.

**Note:** Missing values can silently break or bias calculations (e.g. an average computed while quietly ignoring 30% of rows). Finding them early means *you* decide what to do about them, instead of being surprised by a wrong result later.

**Hint:**
You can use isna() and sum() methods.
```

## Question 3 — Investigate *why* values are missing

`retweet_count`, `reply_count`, `like_count`, `quote_count` are missing for the same rows. Check whether the missing values are concentrated in specific `author_id`s, or spread randomly.

**Note:** Not all missing data is the same. Randomly missing data behaves very differently from data missing for a systematic reason (e.g. certain accounts hide their stats). What is your findings?

**Hint:**
use isna()

## Question 4 — Handle the missing values

Decide whether to `fillna()` or `dropna()` the rows with missing engagement counts, and apply it.

**Note:** `dropna()` and `fillna()` produce different datasets and can lead to different conclusions. Picking one without a reason is a common way analyses might lead to errors.

**Hint:**
Use `dropna()` and `fillna()`
```

## Question 5 — Fix the date column

`created_at` is stored as text. Convert it to datetime, then keep only tweets posted from 1 September 2024 onwards.

**Note:** A text column that *looks* like a date still behaves like text: you can't reliably filter, sort chronologically, or compute time differences on it. This is one of the most common real-world data analysis issue — always check dtype before assuming a date column.

**Hint:**
Use pd.to_datetime() to convert the datatype of a particular column.

## Question 6 — Recategorize a messy column

`tweets['topic_raw']` has many specific categories. Write a function that groups them into 4 broader buckets and apply it to create a new `topic` column.

**Note:** Real category columns are often too granular to analyze usefully (dozens of labels, some with only a handful of rows). Grouping them into a few meaningful buckets is what makes group comparisons (like Question 9) actually readable and meaningful.

**Hint:** (you can customize this function)
```python
def recategorize(topic):
    if topic in ['Climate Policy', 'Climate Denial Debate']:
        return 'Policy & Society'
    elif topic in ['Extreme Weather', 'Wildlife & Conservation']:
        return 'Environment & Nature'
    elif topic in ['Renewable Energy', 'Green Technology', 'Corporate Sustainability']:
        return 'Technology & Business'
    else:
        return 'Other'

tweets['topic'] = tweets['topic_raw'].apply(recategorize)
```

## Question 7 — Load the sentiment file and fix its dtype

Load `synthetic_sentiment.csv` into `sentiment`. Check the dtypes of `positive`, `negative`, `neutral` — they should be numeric but aren't. Investigate why, then clean them.

**Note:** A column that 'looks numeric' can still load as text if even one value doesn't parse (a typo, a placeholder like 'unk'). If you don't catch this, `.mean()` and similar functions would not work or silently give you a wrong result.

**Hint:**

Use sentiment['positive'].unique()  to look for non-numeric values and pd.to_numeric() to fix the datatype.


## Question 8 — Merge tweets and sentiment

Check `tweet_id` is unique in both dataframes, then merge `tweets` and `sentiment` on `tweet_id` using `how='left'`, `'right'`, `'inner'`, `'outer'` and compare the resulting lengths.

**Note:** If a merge key isn't unique, a merge can duplicate rows and inflate your dataset without throwing an error. Always check uniqueness first. And picking the wrong `how=` is one of the most common real-world data bugs: it either drops rows you needed or inflates count. Comparing lengths across all four is a fast sanity check e.g. if `inner` is much smaller than `left`, a lot of your tweets simply don't have a sentiment score.

**Hint:**
You can use something like this:
```python
tweets['tweet_id'].is_unique, sentiment['tweet_id'].is_unique
for how in ['left', 'right', 'inner', 'outer']:
    print(how, len(tweets.merge(sentiment, on='tweet_id', how=how)))
```

## Question 9 — Group and summarize

Group `merged` by `topic` and compute mean and standard deviation of the engagement columns. Use `.transpose()` for readability.

**Note:** Grouping is how you analyze large data report your finding, like 'topic A gets more engagement than topic B'. 

**Hint:**
```python
merged.groupby('topic')
```

## Question 10 — Aggregate sentiment score & sample check

Create one column `sentiment_compound` that combines `positive` and `negative` into a single score, describe it (mean/SD), then take a random sample of 15 rows and look at `text` next to the score.

**Note:** Combining several columns into one score is common (composite indices, summary scores) but it hides assumptions about how the pieces should combine. Manually checking a sample against the raw text is how you sanity-check that your formula actually behaves the way you think it does, before trusting it at scale.

**Hint:**
Example merging 
```python
merged['sentiment_compound'] = merged['positive'] + merged['negative']
```

## Question 11 — Understand the users table

Load `synthetic_users_raw.csv` into `users_raw`. Compare `len(users_raw)` to `users_raw['id'].nunique()` and check `.value_counts()` on `id` to see why they differ.

**Note:** Data pulled from APIs often have similar structural duplication built in. If you don't recognize it, you'll miscount things (e.g. think you have more distinct users than you do) and later merges will behave unexpectedly.


## Question 12 — Create a unique users table

Create `users_unique` with exactly one row per user `id`.

**Note:** This is the fix for Question 11 — but `drop_duplicates()` can silently throw away rows you actually wanted if you don't specify the right `subset`. Always be clear on *what makes a row a duplicate* before you call it, not just the syntax.

**Hint:**
You can use drop_duplicates() 

## Question 13 — Data minimization

Merge `merged` with `users_unique` (left merge, tweets on the left) to add author info. Then create `df_min` keeping only columns relevant to: *does sentiment predict engagement, controlling for follower count?*

**Note:** Keeping every column makes a dataset harder to work with, and with personal data it's a privacy risk. Align your column selection to your actual research question is good practice generally.

**Hint:**
```python
df = merged.merge(users_unique, how='left', left_on='author_id', right_on='id', suffixes=('_tweet', '_user'))
df_min = df[['tweet_id', 'username', 'text', 'sentiment_compound', 'like_count', 'retweet_count', 'followers_count', 'verified']]
```

In [35]:
df_min = df[['tweet_id', 'username', 'text', 'sentiment_compound',
             'like_count', 'retweet_count', 'followers_count', 'verified']]
df_min.head()

,tweet_id,username,text,sentiment_compound,like_count,retweet_count,followers_count,verified
0,1700000000000000091,urbandaily8086_8,RT @solarhub3140_19: @sunnytalks3920_48 My res...,1.0,8.0,4.0,156,False
1,1700000000000000273,bluedaily2168_17,Op-ed: what we're getting wrong about wildlife...,-2.0,NaN,NaN,356,False
2,1700000000000000819,sunnyaction8977_47,RT @carbondaily5068_40: Small businesses are a...,NaN,561.0,5.0,125,False
3,1700000000000000910,windylab453_37,Op-ed: what we're getting wrong about wildlife...,3.0,18.0,3.0,65,False
4,1700000000000001092,polarnow4340_7,"Just read a great thread on green technology, ...",1.0,21.0,3.0,76,False


## Question 14 — Pseudonymize the authors

`df_min` still has `username`. Build a lookup of unique usernames with a sequential `pseudoID`, merge it in, then delete `username`.

**Note:** Pseudonymization keeps analysis possible (you can still tell if the same person posted twice) while removing the direct identifier.

**Hint:**
Functions you'll need -
* drop_duplicates() — removes duplicate rows, keeping one row per unique username
range(len(df)) — generates sequential numbers 0, 1, 2, ..., one per row
* merge(other_df, how='left', on='column_name') — combines two dataframes by matching rows on a shared column
* del df['column_name'] — removes a column entirely

## Question 15 — Anonymize mentions in the tweet text

Replace every `@username` mention inside `df_min['text']` with the placeholder `@mention`.

**Note:** Even after removing usernames from the users table, free text can still contain identifying information — here, other people's handles. This is a reminder that privacy risks hide in unstructured fields too, not just in obviously-named columns like `username`.

**Hint:**
```python
df_min['text'] = df_min['text'].replace(to_replace=r'@\S+', value='@mention', regex=True)
```

In [54]:
# ANSWER ()
df_min['text'] = df_min['text'].replace(to_replace=r'@\S+', value='@mention', regex=True)
df_min['text'].head()

0    RT @mention @mention My research this year has...
1    RT @mention Small businesses are adapting fast...
2    Op-ed: what we're getting wrong about wildlife...
3    Just read a great thread on green technology, ...
4    Can we talk about how wildlife conservation af...
Name: text, dtype: str

# Bonus Questions

## Bonus 1 — Detect retweets and compare engagement

Tweets that start with `RT @username:` are retweets, not original content. Create a boolean column `is_retweet` on `tweets`, then compare mean engagement (`like_count`, `retweet_count`) between retweets and originals.

**Note:** Retweets and original posts are fundamentally different content (an endorsement vs. an authored opinion). Lumping them together can quietly distort an analysis — here, retweets and originals actually have very different average engagement.

**Hint:**
```python
tweets['is_retweet'] = tweets['text'].str.match(r'^RT @\w+:')
tweets.groupby('is_retweet')[['like_count', 'retweet_count']].mean()
```

## Bonus 2 — Extract and rank hashtags

Write a function that extracts all hashtags from a tweet's text as a list, apply it to create a `hashtags` column, then use `.explode()` to find the 10 most common hashtags overall.

**Note:** Regular expressions are the standard tool for pulling structured pieces (hashtags, mentions, emails, URLs) out of free text — a skill that transfers directly to almost any text dataset you'll work with, not just Twitter data.

**Hint:**
```python
def extract_hashtags(text):
    return re.findall(r'#\w+', text)

tweets['hashtags'] = tweets['text'].apply(extract_hashtags)
tweets.explode('hashtags')['hashtags'].value_counts().head(10)
```

## Bonus 3 — Follower-normalized engagement rate

Using `df` (tweets merged with user info), compute `engagement_rate = like_count / followers_count` for each tweet. Handle the case where `followers_count` is 0 (avoid dividing by zero). Then find the 5 authors with the highest *average* engagement rate.

**Note:** Raw counts favor big accounts; normalizing by followers lets you fairly compare a small account to a large one. Handling the divide-by-zero case is also a realistic reminder that real data has edge cases that will crash naive code if you don't plan for them.

**Hint:**
```python
df['engagement_rate'] = df['like_count'] / df['followers_count'].replace(0, pd.NA)
df.groupby('username')['engagement_rate'].mean().sort_values(ascending=False).head(5)
```